In [ ]:
import joblib
import pandas as pd

# Load the saved model
loaded_model = joblib.load("model.joblib")
print("✅ Model loaded successfully!")

In [ ]:
import os

folders = [
    "app/core",
    "app/api",
    "app/services",
    "app/middleware",
    "models",
    "monitoring",
]

for f in folders:
    os.makedirs(f, exist_ok=True)

for pkg in [
    "app",
    "app/core",
    "app/api",
    "app/services",
    "app/middleware",
]:
    open(os.path.join(pkg, "__init__.py"), "a").close()

print("✅ Structure created!")

✅ Structure created!


In [ ]:
%%bash
git init
echo "__pycache__/\n*.pyc\n*.pyo\n*.pyd\n*.db\n.env\nmodels/model.joblib\n.ipynb_checkpoints/" > .gitignore
git add .
git commit -m "Initial commit"
git branch -M main
git remote add origin https://github.com/<your-username>/<PROJECT-NAME>.git
git push -u origin main


Reinitialized existing Git repository in C:/Users/AMAN KUMAR VERMA/Downloads/fastapi_projects/.git/


[main 7b68133] Initial commit
 31 files changed, 1072 insertions(+), 5 deletions(-)
 create mode 100644 app/__init__.py
 create mode 100644 app/__pycache__/__init__.cpython-314.pyc
 create mode 100644 app/__pycache__/main.cpython-314.pyc
 create mode 100644 app/api/__init__.py
 create mode 100644 app/api/__pycache__/__init__.cpython-314.pyc
 create mode 100644 app/api/__pycache__/routes_auth.cpython-314.pyc
 create mode 100644 app/api/dependencies.py
 create mode 100644 app/api/exceptions.py
 create mode 100644 app/api/routes_auth.py
 create mode 100644 app/api/routes_predict.py
 create mode 100644 app/core/__init__.py
 create mode 100644 app/core/__pycache__/__init__.cpython-314.pyc
 create mode 100644 app/core/__pycache__/security.cpython-314.pyc
 create mode 100644 app/core/config.py
 create mode 100644 app/core/security.py
 create mode 100644 app/git init.py
 create mode 100644 app/main.py
 create mode 100644 app/middleware/__init__.py
 create mode 100644 app/middleware/logging_mid

bash: line 6: your-username: No such file or directory
To https://github.com/amankumarverma2703akv-dot/fastapi_projects.git
   0c1e832..7b68133  main -> main


branch 'main' set up to track 'origin/main'.


In [ ]:
%%writefile app/core/config.py
from pydantic_settings import BaseSettings

class Settings(BaseSettings):
    PROJECT_NAME: str = "BFSI1"
    SECRET_KEY: str = "super-secret-key-123"
    ALGORITHM: str = "HS256"
    ACCESS_TOKEN_EXPIRE_MINUTES: int = 30
    REDIS_URL: str = "redis://redis:6379"

    class Config:
        env_file = ".env"

settings = Settings()

Overwriting app/core/config.py


In [ ]:
%%writefile app/core/security.py
from datetime import datetime, timedelta
from jose import jwt, JWTError
from app.core.config import settings

def create_access_token(data: dict, expires_delta: timedelta = None):
    to_encode = data.copy()
    expire = datetime.utcnow() + (expires_delta or timedelta(minutes=settings.ACCESS_TOKEN_EXPIRE_MINUTES))
    to_encode.update({"exp": expire})
    return jwt.encode(to_encode, settings.SECRET_KEY, algorithm=settings.ALGORITHM)

def verify_token(token: str):
    try:
        payload = jwt.decode(token, settings.SECRET_KEY, algorithms=[settings.ALGORITHM])
        return payload
    except JWTError:
        return None


Overwriting app/core/security.py


In [ ]:
%%writefile app/api/dependencies.py
from fastapi import Depends, HTTPException, status
from fastapi.security import OAuth2PasswordBearer
from app.core.security import verify_token

oauth2_scheme = OAuth2PasswordBearer(tokenUrl="login")

def get_current_user(token: str = Depends(oauth2_scheme)):
    payload = verify_token(token)
    if payload is None:
        raise HTTPException(status_code=status.HTTP_401_UNAUTHORIZED, detail="Invalid token")
    return payload


Overwriting app/api/dependencies.py


In [ ]:
%%writefile app/api/exceptions.py
from fastapi.responses import JSONResponse
from fastapi import Request

async def http_exception_handler(request: Request, exc):
    return JSONResponse(status_code=exc.status_code, content={"detail": exc.detail})


Overwriting app/api/exceptions.py


In [ ]:
%%writefile app/api/routes_auth.py
from fastapi import APIRouter, Depends
from fastapi.security import OAuth2PasswordRequestForm
from app.core.security import create_access_token

router = APIRouter()

@router.post("/login")
def login(form_data: OAuth2PasswordRequestForm = Depends()):
    if form_data.username != "admin" or form_data.password != "password":
        return {"error": "Invalid credentials"}
    token = create_access_token({"sub": form_data.username})
    return {"access_token": token, "token_type": "bearer"}


Overwriting app/api/routes_auth.py


In [ ]:
%%writefile app/services/redis_cache.py
import redis
from app.core.config import settings

redis_client = redis.Redis.from_url(settings.REDIS_URL, decode_responses=True)

def get_cache(key: str):
    return redis_client.get(key)

def set_cache(key: str, value: str, expire: int = 300):
    redis_client.set(key, value, ex=expire)


Overwriting app/services/redis_cache.py


In [ ]:
%%writefile app/services/model_service.py
import joblib

model, scaler = joblib.load("models/model.joblib")

def predict(features: list):
    scaled = scaler.transform([features])
    return int(model.predict(scaled)[0])


Overwriting app/services/model_service.py


In [ ]:
%%writefile app/api/routes_predict.py
from fastapi import APIRouter, Depends
from app.api.dependencies import get_current_user
from app.services.redis_cache import get_cache, set_cache
from app.services.model_service import predict

router = APIRouter()

@router.post("/predict")
def predict_endpoint(features: list, user: dict = Depends(get_current_user)):
    key = str(features)
    cached = get_cache(key)
    if cached:
        return {"prediction": int(cached), "cached": True}
    result = predict(features)
    set_cache(key, result)
    return {"prediction": result, "cached": False}


Overwriting app/api/routes_predict.py


In [ ]:
%%writefile app/middleware/logging_middleware.py
from starlette.middleware.base import BaseHTTPMiddleware
import logging

logger = logging.getLogger("uvicorn")

class LoggingMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request, call_next):
        logger.info(f"Request: {request.method} {request.url}")
        response = await call_next(request)
        logger.info(f"Response status: {response.status_code}")
        return response


Overwriting app/middleware/logging_middleware.py


In [ ]:
%%writefile app/main.py
from fastapi import FastAPI
from app.api import routes_auth, routes_predict
from app.middleware.logging_middleware import LoggingMiddleware

app = FastAPI()
app.include_router(routes_auth.router)
app.include_router(routes_predict.router)
app.add_middleware(LoggingMiddleware)


Overwriting app/main.py


In [ ]:
import nest_asyncio
import uvicorn

nest_asyncio.apply()
uvicorn.run("app.main:app", host="0.0.0.0", port=8000, reload=True)


INFO:     Will watch for changes in these directories: ['c:\\Users\\AMAN KUMAR VERMA\\Downloads\\fastapi_projects']
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [13900] using StatReload
INFO:     Stopping reloader process [13900]


In [ ]:
%%writefile monitoring/prometheus.yml
global:
  scrape_interval: 15s
scrape_configs:
  - job_name: 'fastapi'
    static_configs:
      - targets: ['app:8000']


Overwriting monitoring/prometheus.yml


In [ ]:
%%writefile docker-compose.yml
services:
  app:
    build:
      context: .
      dockerfile: docker/Dockerfile
    ports:
      - "8000:8000"
    environment:
      - REDIS_URL=redis://redis:6379
      - SECRET_KEY=mysecretkey
    depends_on:
      - redis

  redis:
    image: redis:alpine
    ports:
      - "6379:6379"

Overwriting docker-compose.yml


In [ ]:
import os

os.makedirs("docker", exist_ok=True)
print("✅ 'docker' directory ready!")

✅ 'docker' directory ready!


In [ ]:
%%writefile docker/Dockerfile
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8000

CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]

Overwriting docker/Dockerfile


In [ ]:
%%writefile requirements.txt
fastapi
uvicorn
python-jose
pydantic
redis
scikit-learn
joblib
prometheus-client

Overwriting requirements.txt


In [ ]:
! docker ps - a

docker: 'docker ps' accepts no arguments

Usage:  docker ps [OPTIONS]

Run 'docker ps --help' for more information


In [ ]:
! docker compose up -d --build

unable to get image 'redis:alpine': failed to connect to the docker API at npipe:////./pipe/dockerDesktopLinuxEngine; check if the path is correct and if the daemon is running: open //./pipe/dockerDesktopLinuxEngine: The system cannot find the file specified.


In [ ]:
%%writefile render.yaml
services:
  - type: web
    name: fastapi-ml-app
    env: python
    buildCommand: "pip install -r requirements.txt"
    startCommand: "uvicorn app.main:app --host 0.0.0.0 --port 8000"
envVars:
  - key: REDIS_URL
    value: "redis://<your-free-redis-instance>"


Overwriting render.yaml


In [ ]:
import requests

# 1. Test Healthcheck Endpoint
response = requests.get("http://127.0.0.1:8000/")
print("1. Healthcheck Output:", response.json())

# 2. Test Login Endpoint
login_data = {"username": "admin", "password": "password"}

auth_response = requests.post(
    "http://127.0.0.1:8000/auth/login", data=login_data
)
print("2. Login Output:", auth_response.json())

ConnectionError: HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: / (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=8000): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))

In [ ]:
! docker compose logs app

app-1  | Traceback (most recent call last):
app-1  |   File "/usr/local/bin/uvicorn", line 8, in <module>
app-1  |     sys.exit(main())
app-1  |              ^^^^^^
app-1  |   File "/usr/local/lib/python3.11/site-packages/click/core.py", line 1569, in __call__
app-1  |     return self.main(*args, **kwargs)
app-1  |            ^^^^^^^^^^^^^^^^^^^^^^^^^^
app-1  |   File "/usr/local/lib/python3.11/site-packages/click/core.py", line 1490, in main
app-1  |     rv = self.invoke(ctx)
app-1  |          ^^^^^^^^^^^^^^^^
app-1  |   File "/usr/local/lib/python3.11/site-packages/click/core.py", line 1353, in invoke
app-1  |     return ctx.invoke(self.callback, **ctx.params)
app-1  |            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
app-1  |   File "/usr/local/lib/python3.11/site-packages/click/core.py", line 907, in invoke
app-1  |     return callback(*args, **kwargs)
app-1  |            ^^^^^^^^^^^^^^^^^^^^^^^^^
app-1  |   File "/usr/local/lib/python3.11/site-packages/uvicorn/main.py", line 440,

time="2026-08-12T20:44:10+05:30" level=warning msg="c:\\Users\\AMAN KUMAR VERMA\\Downloads\\fastapi_projects\\docker-compose.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"


pydantic.errors.PydanticImportError: `BaseSettings` has been moved to the `pydantic-settings` package.
```[cite: 2]

---

## ❌ The 3 Mistakes in Your Notebook

### 1. Pydantic V2 Import Error (`app/core/config.py`)
In **Cell 3**, you wrote `from pydantic import BaseSettings`[cite: 2]. In modern Pydantic (v2), `BaseSettings` was removed from the main `pydantic` package and moved to `pydantic_settings`.

### 2. Missing Dependency in `requirements.txt`
In **Cell 32**, `pydantic-settings` was missing from your `requirements.txt`[cite: 2].

### 3. Syntax Error in Docker command
In **Cell 33**, you ran `! docker ps - a` (with a space between `-` and `a`), which gave a Docker syntax error[cite: 2].

---

## 🛠️ The Clean Fix

Run these 3 cells in your notebook to update the code and restart Docker:

### Step 1: Update `app/core/config.py`
```python
%%writefile app/core/config.py
from pydantic_settings import BaseSettings

class Settings(BaseSettings):
    PROJECT_NAME: str = "BFSI1"
    SECRET_KEY: str = "super-secret-key-123"
    ALGORITHM: str = "HS256"
    ACCESS_TOKEN_EXPIRE_MINUTES: int = 30
    REDIS_URL: str = "redis://redis:6379"

    class Config:
        env_file = ".env"

settings = Settings()